# Tokenization (BPE, WordPiece, SentencePiece) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Subword tokenization as compression with an open-vocabulary safety valve
The examples learn BPE merges, compare character and word tokenization, show unknown-word handling, mark WordPiece continuations, and use unigram probabilities.

In [ ]:
words=['low','lower','newest','widest']; vocab=[list(w)+['</w>'] for w in words]
pairs={}
for w in vocab:
    for a,b in zip(w,w[1:]): pairs[(a,b)]=pairs.get((a,b),0)+1
labels=[str(k) for k in pairs]; vals=list(pairs.values())
plt.figure(figsize=(7,3)); plt.bar(labels, vals); plt.xticks(rotation=45); plt.title('BPE counts adjacent pairs')
assert max(pairs, key=pairs.get)==('l','o')

In [ ]:
toks=list('low</w>'.replace('</w>','_'))
merged=['lo','w','_']
plt.figure(figsize=(5,3)); plt.bar(['chars','after lo merge'],[len(toks),len(merged)]); plt.title('A merge shortens frequent fragments')
assert len(merged)==3 and 'lo' in merged

In [ ]:
vocab={'low','lower'}; word='lowest'
word_tokens=[word if word in vocab else '<unk>']
sub_tokens=['low','est']
plt.figure(figsize=(5,3)); plt.bar(['word-level','subword'],[len(word_tokens),len(sub_tokens)]); plt.title('Subwords avoid unknown whole words')
assert word_tokens==['<unk>'] and sub_tokens[0]=='low'

In [ ]:
pieces=['play','##ing','play','##er','un','##play','##able']; cont=[p.startswith('##') for p in pieces]
plt.figure(figsize=(7,2)); plt.imshow([cont], aspect='auto', cmap='Oranges'); plt.xticks(range(len(pieces)),pieces,rotation=45); plt.yticks([]); plt.title('WordPiece marks continuations')
assert sum(cont)==4

In [ ]:
segs=[['the','re'],['ther','e'],['t','here']]; probs={('the','re'):0.32,('ther','e'):0.18,('t','here'):0.05}
scores=[probs[tuple(s)] for s in segs]
plt.figure(figsize=(6,3)); plt.bar(['+'.join(s) for s in segs],scores); plt.title('SentencePiece unigram chooses probable segmentation')
assert segs[int(np.argmax(scores))]==['the','re']